# ASL-EMG — Exploring One Random Subject

This notebook loads the **ASL-EMG** dataset and walks through the data of **one randomly chosen subject**: what was recorded, the signal schema, and a set of visualizations of the EMG and IMU streams.

## What the dataset is

Sign-language gestures captured from **two Myo armbands** (one per forearm). Each armband streams:

- **8 EMG channels** — surface electromyography, integer ADC counts (muscle activation).
- **IMU** — accelerometer (`AX,AY,AZ`, g), gyroscope (`GX,GY,GZ`, deg/s), orientation (`OR,OP,OY` = roll/pitch/yaw).

Columns are suffixed **`L`** (left arm) and **`R`** (right arm). Each CSV is **one recording** = a header row plus **50 timesteps** (`Counter` 1..50), so a sample is a `50 x 34` feature time series.

Two collections live inside `wgswcr8z24-2.zip`:
- `10_ASL_Signs.zip` -> users `User1..User10` (User5 absent), 20 distinct **ASL word** signs.
- `15health_signs_5singers.zip` -> `HealthSigns_User1..5`, 15 **health-phrase** signs.

A recording's **label** is the filename prefix before the underscore-id (e.g. `headache_145367335.csv` -> `headache`).

> Runs on **Carnets Plus** (on-device iOS Jupyter): only the standard library + numpy/pandas/matplotlib are used, and CSVs are read **directly out of the nested zips** so nothing has to be extracted on the device.

## 1. Setup

Put `wgswcr8z24-2.zip` (or the two inner zips) somewhere the notebook can find it — the same folder as this notebook works. The loader searches the current folder and a few parents.

In [ ]:
import io, os, zipfile, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Set a seed for a reproducible 'random' subject; comment out for a fresh pick each run.
SEED = None
if SEED is not None:
    random.seed(SEED)

## 2. Locate and open the data

We open the outer zip in memory and, inside it, the two inner zips — all via the stdlib `zipfile`. No files are written to disk.

In [ ]:
def _find(names, max_up=4):
    """Search cwd and up to `max_up` parent dirs (recursively) for the first matching filename."""
    root = os.getcwd()
    for _ in range(max_up + 1):
        for dirpath, _dirs, files in os.walk(root):
            for n in names:
                if n in files:
                    return os.path.join(dirpath, n)
        root = os.path.dirname(root)
    return None

def open_inner_zips():
    """Return {inner_zip_name: ZipFile}. Works whether you have the outer zip or the two inner zips."""
    inners = {}
    outer_path = _find(['wgswcr8z24-2.zip'])
    if outer_path:
        print('Found outer zip:', outer_path)
        outer = zipfile.ZipFile(outer_path)
        for n in outer.namelist():
            if n.endswith('.zip'):
                inners[os.path.basename(n)] = zipfile.ZipFile(io.BytesIO(outer.read(n)))
    else:
        for n in ['10_ASL_Signs.zip', '15health_signs_5singers.zip']:
            p = _find([n])
            if p:
                print('Found inner zip:', p)
                inners[n] = zipfile.ZipFile(p)
    if not inners:
        raise FileNotFoundError('Could not find wgswcr8z24-2.zip or the inner zips. '
                                'Place the dataset next to this notebook.')
    return inners

INNERS = open_inner_zips()
list(INNERS.keys())

## 3. Build an index of every recording

Each CSV path looks like `all/User10/bird_143993937.csv` or `health_signs/HealthSigns_User1/headache_145367335.csv`. We parse out **collection / user / sign** for every file. One label typo is normalized: `continuoslyforanhour` -> `continuouslyforanhour`.

In [ ]:
def sign_from_filename(fname):
    stem = os.path.splitext(os.path.basename(fname))[0]
    sign = stem.rsplit('_', 1)[0]            # drop the trailing capture id
    return 'continuouslyforanhour' if sign == 'continuoslyforanhour' else sign

rows = []
for zip_name, zf in INNERS.items():
    for path in zf.namelist():
        if not path.endswith('.csv'):
            continue
        parts = path.split('/')
        if len(parts) < 3:
            continue
        rows.append({'zip': zip_name, 'user': parts[1],
                     'sign': sign_from_filename(path), 'path': path})

index = pd.DataFrame(rows)
print('Total recordings:', len(index))
print('Subjects (zip, user):', index.groupby(['zip', 'user']).ngroups)
index.head()

## 4. Pick one random subject

A *subject* is a `(collection, user)` pair. We pick one at random and keep only its recordings.

In [ ]:
subjects = list(index.groupby(['zip', 'user']).groups.keys())
chosen_zip, chosen_user = random.choice(subjects)
subj = index[(index['zip'] == chosen_zip) & (index['user'] == chosen_user)].reset_index(drop=True)

print('Chosen subject :', chosen_user, ' from ', chosen_zip)
print('Recordings     :', len(subj))
print('Distinct signs :', subj['sign'].nunique())
subj['sign'].value_counts().sort_index()

## 5. A small CSV reader

Reads one recording straight from its zip into a DataFrame. `drop_warmup` removes the first rows where **all EMG channels are 0** — the Myo EMG needs a sample or two to start streaming while the IMU is already valid.

In [ ]:
EMG_L = [f'EMG{i}L' for i in range(8)]
EMG_R = [f'EMG{i}R' for i in range(8)]
EMG_ALL = EMG_L + EMG_R

def _zip_for(path):
    for zip_name, zf in INNERS.items():
        if path in zf.namelist():
            return zip_name
    raise KeyError(path)

def read_recording(path, drop_warmup=True):
    raw = INNERS[_zip_for(path)].read(path)
    df = pd.read_csv(io.BytesIO(raw))
    if drop_warmup:
        active = (df[EMG_ALL] != 0).any(axis=1)
        df = df[active].reset_index(drop=True)
    return df

example = subj.iloc[random.randrange(len(subj))]
df = read_recording(example['path'])
print('Example recording:', example['path'])
print('Sign            :', example['sign'])
print('Shape after warm-up drop:', df.shape)
df.head()

## 6. EMG — both arms for one recording

The 8 EMG channels per arm over the recording. EMG is roughly zero-mean noise that bursts when the underlying muscles contract, so the interesting structure is the **amplitude envelope** across the gesture.

In [ ]:
t = df['Counter'].values
fig, axes = plt.subplots(8, 2, figsize=(11, 12), sharex=True)
for i in range(8):
    axes[i, 0].plot(t, df[f'EMG{i}L'], lw=0.9, color='C0')
    axes[i, 1].plot(t, df[f'EMG{i}R'], lw=0.9, color='C3')
    axes[i, 0].set_ylabel(f'EMG{i}')
axes[0, 0].set_title('Left arm')
axes[0, 1].set_title('Right arm')
axes[7, 0].set_xlabel('timestep (Counter)')
axes[7, 1].set_xlabel('timestep (Counter)')
fig.suptitle(f"EMG — {chosen_user}, sign '{example['sign']}'", y=0.995)
fig.tight_layout()
plt.show()

## 7. IMU — motion of each armband

The accelerometer, gyroscope, and orientation streams describe how the forearms moved during the sign. Unlike EMG these are smooth, low-frequency trajectories.

In [ ]:
groups = [('Accelerometer (g)', ['AX', 'AY', 'AZ']),
          ('Gyroscope (deg/s)', ['GX', 'GY', 'GZ']),
          ('Orientation (roll/pitch/yaw)', ['OR', 'OP', 'OY'])]

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True)
for col, (arm, suffix) in enumerate([('Left', 'L'), ('Right', 'R')]):
    for row, (title, base) in enumerate(groups):
        ax = axes[row, col]
        for b in base:
            ax.plot(t, df[f'{b}{suffix}'], lw=1.1, label=b)
        if col == 0:
            ax.set_ylabel(title, fontsize=9)
        ax.legend(fontsize=7, ncol=3, loc='upper right')
    axes[0, col].set_title(f'{arm} arm')
axes[2, 0].set_xlabel('timestep (Counter)')
axes[2, 1].set_xlabel('timestep (Counter)')
fig.suptitle(f"IMU — {chosen_user}, sign '{example['sign']}'", y=0.995)
fig.tight_layout()
plt.show()

## 8. Which muscles fire for which sign?

For every recording of this subject we compute the **mean rectified EMG** (`mean(|EMG|)`) per channel, then average across repetitions of each sign. The heatmap shows a muscle-activation fingerprint per sign — distinct rows mean the signs are EMG-separable, which is what a classifier exploits.

In [ ]:
per_sign = {}
for sign, grp in subj.groupby('sign'):
    mats = []
    for p in grp['path']:
        d = read_recording(p)
        if len(d):
            mats.append(d[EMG_ALL].abs().mean().values)
    if mats:
        per_sign[sign] = np.mean(mats, axis=0)

act = pd.DataFrame(per_sign, index=EMG_ALL).T   # rows=signs, cols=16 channels
act = act.sort_index()

fig, ax = plt.subplots(figsize=(11, 0.45 * len(act) + 2))
im = ax.imshow(act.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(EMG_ALL)))
ax.set_xticklabels(EMG_ALL, rotation=90, fontsize=7)
ax.set_yticks(range(len(act)))
ax.set_yticklabels(act.index, fontsize=8)
ax.set_xlabel('EMG channel (left arm | right arm)')
ax.set_title(f'Mean rectified EMG per sign — {chosen_user}')
ax.axvline(7.5, color='w', lw=1)   # divide left vs right arm
fig.colorbar(im, ax=ax, label='mean |EMG|')
fig.tight_layout()
plt.show()

## 9. Repetition consistency for one sign

How repeatable is a single sign? We overlay the EMG envelope (RMS across all 16 channels at each timestep) for every repetition of one sign. Tight overlap = consistent execution; spread = natural variability the model must tolerate.

In [ ]:
target_sign = subj['sign'].value_counts().idxmax()   # the sign with the most reps
reps = subj[subj['sign'] == target_sign]['path'].tolist()

fig, ax = plt.subplots(figsize=(10, 4))
for j, p in enumerate(reps):
    d = read_recording(p)
    env = np.sqrt((d[EMG_ALL].astype(float) ** 2).mean(axis=1))   # RMS over channels
    ax.plot(d['Counter'], env, lw=1.3, label=f'rep {j + 1}')
ax.set_xlabel('timestep (Counter)')
ax.set_ylabel('RMS EMG across 16 channels')
ax.set_title(f"Repetitions of '{target_sign}' — {chosen_user}")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Notes

- **Reproducibility:** set `SEED` in cell 1 to always land on the same subject.
- **Warm-up rows:** `read_recording(..., drop_warmup=False)` keeps the leading all-zero-EMG rows if you want the raw 50 timesteps.
- **Sampling:** Myo EMG is ~200 Hz and IMU ~50 Hz, but here each file is fixed at 50 stored timesteps; treat `Counter` as the time axis, not seconds.
- **Carnets:** everything above uses only stdlib + numpy/pandas/matplotlib, all bundled. No extraction, no extra installs.